# Random Walk Feature Encoder: Testing and Evaluation

This notebook tests the new Random Walk Feature Encoder on Titanic data.

## Key Questions:
1. How well does a linear first-layer NN recover original relationships?
2. What is the privacy protection against reconstruction attacks?
3. How do different parameter settings affect utility vs privacy?
4. How should binary features be handled?

In [ ]:
# Setup: Imports
import sys
from pathlib import Path

# Add project root to path
notebook_path = Path().resolve()
if notebook_path.name == 'notebooks':
    project_root = notebook_path.parent
else:
    project_root = notebook_path

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# Force reload to pick up latest encoder changes (especially output_noise parameter)
import importlib
if 'quagua.encoders.random_walk_encoder' in sys.modules:
    importlib.reload(sys.modules['quagua.encoders.random_walk_encoder'])
if 'quagua.encoders' in sys.modules:
    importlib.reload(sys.modules['quagua.encoders'])

from quagua.encoders import RandomWalkFeatureEncoder
from quagua.data_utils import load_titanic_data
from quagua.evaluation import PrivacyAITestFramework

np.random.seed(42)

print("="*80)
print("RANDOM WALK FEATURE ENCODER: TESTING AND EVALUATION")
print("="*80)

## 1. Load Data and Setup

In [ ]:
# Load Titanic data
X, y, feature_names = load_titanic_data(include_continuous=True)
print(f"Loaded {X.shape[0]} samples with {X.shape[1]} features")
print(f"Features: {feature_names}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples")

## 2. Enhanced Model Training Functions

We'll use improved models with better hyperparameters for a fair comparison.

In [ ]:
# Enhanced model training functions for better comparison
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

def train_models(X_train, y_train, X_test, y_test, verbose=True):
    """
    Train multiple models with improved hyperparameters for comprehensive comparison.
    Returns best model and all results.
    """
    results = {}
    
    # 1. MLP - Larger network, more iterations
    if verbose:
        print("Training MLP (enhanced)...")
    mlp = MLPClassifier(
        hidden_layer_sizes=(128, 64, 32),  # Larger network
        activation='relu',
        solver='adam',
        max_iter=2000,  # More iterations
        alpha=0.001,  # L2 regularization
        learning_rate='adaptive',
        learning_rate_init=0.001,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20,
        tol=1e-4
    )
    mlp.fit(X_train, y_train)
    y_pred_mlp = mlp.predict(X_test)
    y_proba_mlp = mlp.predict_proba(X_test)[:, 1]
    
    results['MLP'] = {
        'model': mlp,
        'accuracy': accuracy_score(y_test, y_pred_mlp),
        'f1': f1_score(y_test, y_pred_mlp),
        'auc': roc_auc_score(y_test, y_proba_mlp),
        'predictions': y_pred_mlp,
        'probabilities': y_proba_mlp
    }
    
    # 2. Random Forest - More trees, better parameters
    if verbose:
        print("Training Random Forest (enhanced)...")
    rf = RandomForestClassifier(
        n_estimators=200,  # More trees
        max_depth=15,  # Deeper trees
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_test)
    y_proba_rf = rf.predict_proba(X_test)[:, 1]
    
    results['RandomForest'] = {
        'model': rf,
        'accuracy': accuracy_score(y_test, y_pred_rf),
        'f1': f1_score(y_test, y_pred_rf),
        'auc': roc_auc_score(y_test, y_proba_rf),
        'predictions': y_pred_rf,
        'probabilities': y_proba_rf
    }
    
    # 3. Gradient Boosting - Strong ensemble
    if verbose:
        print("Training Gradient Boosting (enhanced)...")
    gb = GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=5,
        min_samples_split=5,
        random_state=42
    )
    gb.fit(X_train, y_train)
    y_pred_gb = gb.predict(X_test)
    y_proba_gb = gb.predict_proba(X_test)[:, 1]
    
    results['GradientBoosting'] = {
        'model': gb,
        'accuracy': accuracy_score(y_test, y_pred_gb),
        'f1': f1_score(y_test, y_pred_gb),
        'auc': roc_auc_score(y_test, y_proba_gb),
        'predictions': y_pred_gb,
        'probabilities': y_proba_gb
    }
    
    # 4. Logistic Regression - Baseline linear model
    if verbose:
        print("Training Logistic Regression (enhanced)...")
    lr = LogisticRegression(
        max_iter=1000,
        C=1.0,
        penalty='l2',
        solver='lbfgs',
        random_state=42
    )
    lr.fit(X_train, y_train)
    y_pred_lr = lr.predict(X_test)
    y_proba_lr = lr.predict_proba(X_test)[:, 1]
    
    results['LogisticRegression'] = {
        'model': lr,
        'accuracy': accuracy_score(y_test, y_pred_lr),
        'f1': f1_score(y_test, y_pred_lr),
        'auc': roc_auc_score(y_test, y_proba_lr),
        'predictions': y_pred_lr,
        'probabilities': y_proba_lr
    }
    
    # Find best model (by accuracy)
    best_model_name = max(results.keys(), key=lambda k: results[k]['accuracy'])
    best_model = results[best_model_name]['model']
    
    if verbose:
        print(f"\n{'='*60}")
        print("MODEL PERFORMANCE SUMMARY")
        print(f"{'='*60}")
        for name, res in sorted(results.items(), key=lambda x: x[1]['accuracy'], reverse=True):
            print(f"{name:20} | Acc: {res['accuracy']:.4f} | F1: {res['f1']:.4f} | AUC: {res['auc']:.4f}")
        print(f"\nBest model: {best_model_name} (Accuracy: {results[best_model_name]['accuracy']:.4f})")
    
    return {
        'best_model': best_model,
        'best_model_name': best_model_name,
        'all_results': results
    }

## 3. Test Different Random Walk Configurations

Test various Random Walk encoder configurations with improved model training.

## 2. Test Different Configurations

In [ ]:
# Test different configurations (X_train from cell 3)
n_f = X_train.shape[1]
configs = {
    'Random Walk (default)': RandomWalkFeatureEncoder(
        n_steps=None,  # Auto: 2 * n_features
        weight_distribution='beta',
        walk_strategy='random',
        binary_handling='preserve',
        seed=42
    ),
    'Random Walk (uniform weights)': RandomWalkFeatureEncoder(
        n_steps=None,
        weight_distribution='uniform',
        walk_strategy='random',
        binary_handling='preserve',
        seed=42
    ),
    'Random Walk (sequential)': RandomWalkFeatureEncoder(
        n_steps=None,
        weight_distribution='beta',
        walk_strategy='sequential',
        binary_handling='preserve',
        seed=42
    ),
    'Random Walk (normalize binary)': RandomWalkFeatureEncoder(
        n_steps=None,
        weight_distribution='beta',
        walk_strategy='random',
        binary_handling='normalize',
        seed=42
    ),
    'Random Walk (skip binary)': RandomWalkFeatureEncoder(
        n_steps=None,
        weight_distribution='beta',
        walk_strategy='random',
        binary_handling='skip',
        seed=42
    ),
    'Random Walk (more steps, preserve)': RandomWalkFeatureEncoder(
        n_steps=3 * n_f,
        weight_distribution='beta',
        walk_strategy='random',
        binary_handling='preserve',
        seed=42
    ),
    'Random Walk (output noise)': RandomWalkFeatureEncoder(
        n_steps=None,
        weight_distribution='beta',
        walk_strategy='random',
        binary_handling='preserve',
        output_noise=0.08,
        seed=42
    ),
}

print(f"Testing {len(configs)} different configurations...")
print("Note: 'binary' = only features with exactly 2 unique values (Sex, SibSp_bin, Parch_bin).")

## 3. Test with Neural Network (Linear First Layer)

Note: sklearn's MLPClassifier doesn't support a truly linear first layer directly.
In practice, you would use PyTorch/TensorFlow to implement a linear first layer.
For now, we use a standard MLP and analyze if it can learn the inverse transformation.

In [ ]:
# Test each configuration with enhanced model training
results = {}
import warnings
warnings.filterwarnings('ignore')

for name, encoder in configs.items():
    print(f"\n{'='*80}")
    print(f"Testing: {name}")
    print(f"{'='*80}")
    
    # Fit and transform
    X_train_encoded = encoder.fit_transform(X_train)
    X_test_encoded = encoder.transform(X_test)
    
    print(f"Original shape: {X_train.shape}")
    print(f"Encoded shape: {X_train_encoded.shape}")
    
    # Get walk info
    walk_info = encoder.get_walk_info()
    print(f"Walk steps: {walk_info['n_steps']}")
    print(f"Binary features detected: {walk_info['binary_features']}")
    
    # Train enhanced models (multiple model types, better hyperparameters)
    model_results = train_models(X_train_encoded, y_train, X_test_encoded, y_test, verbose=False)
    
    # Use best model results
    best_name = model_results['best_model_name']
    best_results = model_results['all_results'][best_name]
    
    print(f"\nBest Model: {best_name}")
    print(f"  Accuracy: {best_results['accuracy']:.4f}")
    print(f"  F1 Score: {best_results['f1']:.4f}")
    print(f"  ROC AUC: {best_results['auc']:.4f}")
    
    # Store all model results for this encoder
    results[name] = {
        'encoder': encoder,
        'X_train_encoded': X_train_encoded,
        'X_test_encoded': X_test_encoded,
        'best_model_name': best_name,
        'best_model': model_results['best_model'],
        'accuracy': best_results['accuracy'],
        'f1': best_results['f1'],
        'auc': best_results['auc'],
        'all_models': model_results['all_results'],
        'walk_info': walk_info
    }

## 4. Enhanced Baseline (No Encoding)

Train baseline models with the same enhanced training for fair comparison.

In [ ]:
# Baseline: No encoding - use enhanced training
print("\n" + "="*80)
print("BASELINE: No Encoding (Enhanced Training)")
print("="*80)

baseline_results = train_models(X_train, y_train, X_test, y_test, verbose=True)

# Extract best baseline results
baseline_accuracy = baseline_results['all_results'][baseline_results['best_model_name']]['accuracy']
baseline_f1 = baseline_results['all_results'][baseline_results['best_model_name']]['f1']
baseline_auc = baseline_results['all_results'][baseline_results['best_model_name']]['auc']
baseline_best_model_name = baseline_results['best_model_name']

print(f"\nBaseline Best Model: {baseline_best_model_name}")
print(f"  Accuracy: {baseline_accuracy:.4f}")
print(f"  F1 Score: {baseline_f1:.4f}")
print(f"  ROC AUC: {baseline_auc:.4f}")

print(f"Baseline Results:")
print(f"  Accuracy: {baseline_accuracy:.4f}")
print(f"  F1 Score: {baseline_f1:.4f}")
print(f"  ROC AUC: {baseline_auc:.4f}")

# Compare
print("\n" + "="*80)
print("COMPARISON: Encoded vs Baseline")
print("="*80)
print(f"\n{'Method':<45} {'Accuracy':<10} {'F1':<10} {'AUC':<10} {'vs Baseline'}")
print("-"*95)
print(f"{'Baseline (no encoding)':<45} {baseline_accuracy:<10.4f} {baseline_f1:<10.4f} {baseline_auc:<10.4f} {'(baseline)'}")
print(f"{'Stronger baseline (100,50, max_iter=1000)':<45} {strong_acc:<10.4f} {strong_f1:<10.4f} {strong_auc:<10.4f} {'(underfitting check)'}")

for name, result in results.items():
    acc_diff = result['accuracy'] - baseline_accuracy
    f1_diff = result['f1'] - baseline_f1
    auc_diff = result['auc'] - baseline_auc
    
    vs_baseline = f"acc:{acc_diff:+.4f}, f1:{f1_diff:+.4f}, auc:{auc_diff:+.4f}"
    print(f"{name:<45} {result['accuracy']:<10.4f} {result['f1']:<10.4f} {result['auc']:<10.4f} {vs_baseline}")

## 4b. Multi-seed evaluation

Run 5 different random seeds (data split + MLP init) to assess stability of Accuracy, F1, and AUC. Encoder seed is fixed (walk is identical across seeds).

In [ ]:
seeds = [42, 43, 44, 45, 46]
n_f = X.shape[1]
seed_configs = {
    'Baseline': None,
    'Stronger baseline': 'strong',
    'RW default': dict(n_steps=None, weight_distribution='beta', walk_strategy='random', binary_handling='preserve', seed=42),
    'RW normalize': dict(n_steps=None, weight_distribution='beta', walk_strategy='random', binary_handling='normalize', seed=42),
    'RW output noise': dict(n_steps=None, weight_distribution='beta', walk_strategy='random', binary_handling='preserve', output_noise=0.08, seed=42),
    'RW more steps': dict(n_steps=3*n_f, weight_distribution='beta', walk_strategy='random', binary_handling='preserve', seed=42),
}

agg = {k: {'acc': [], 'f1': [], 'auc': []} for k in seed_configs}

for seed in seeds:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    for name, kw in seed_configs.items():
        if kw is None:
            m = MLPClassifier(hidden_layer_sizes=(50,25), max_iter=500, random_state=seed, early_stopping=True, validation_fraction=0.1)
            m.fit(X_tr, y_tr)
        elif kw == 'strong':
            m = MLPClassifier(hidden_layer_sizes=(100,50), max_iter=1000, random_state=seed, early_stopping=True, validation_fraction=0.1)
            m.fit(X_tr, y_tr)
        else:
            enc = RandomWalkFeatureEncoder(**kw)
            X_tr_e, X_te_e = enc.fit_transform(X_tr), enc.transform(X_te)
            m = MLPClassifier(hidden_layer_sizes=(50,25), max_iter=500, random_state=seed, early_stopping=True, validation_fraction=0.1)
            m.fit(X_tr_e, y_tr)
            pred = m.predict(X_te_e)
            proba = m.predict_proba(X_te_e)[:, 1]
            agg[name]['acc'].append(accuracy_score(y_te, pred))
            agg[name]['f1'].append(f1_score(y_te, pred))
            agg[name]['auc'].append(roc_auc_score(y_te, proba))
            continue
        pred = m.predict(X_te)
        proba = m.predict_proba(X_te)[:, 1]
        agg[name]['acc'].append(accuracy_score(y_te, pred))
        agg[name]['f1'].append(f1_score(y_te, pred))
        agg[name]['auc'].append(roc_auc_score(y_te, proba))

print('Multi-seed (seeds=%s) — mean ± std' % seeds)
print('-'*85)
for name in agg:
    a, f, u = np.mean(agg[name]['acc']), np.mean(agg[name]['f1']), np.mean(agg[name]['auc'])
    sa, sf, su = np.std(agg[name]['acc']), np.std(agg[name]['f1']), np.std(agg[name]['auc'])
    print(f"{name:<22}  Acc: {a:.4f}±{sa:.4f}   F1: {f:.4f}±{sf:.4f}   AUC: {u:.4f}±{su:.4f}")

## 5. Privacy Evaluation (Reconstruction Attacks)

In [ ]:
# Evaluate privacy using the framework
framework = PrivacyAITestFramework()

privacy_results = {}

for name, result in results.items():
    print(f"\n{'='*80}")
    print(f"Privacy Evaluation: {name}")
    print(f"{'='*80}")
    
    # Create a simple wrapper to match framework interface
    class EncoderWrapper:
        def __init__(self, encoder):
            self.encoder = encoder
        
        def transform(self, X):
            return self.encoder.transform(X)
    
    wrapper = EncoderWrapper(result['encoder'])
    
    privacy = framework.evaluate_privacy(
        encoder=wrapper,
        X_test=X_test,
        encoder_name=name
    )
    
    privacy_results[name] = privacy
    
    print(f"\nPrivacy Score: {privacy['privacy_score']:.4f}")
    print(f"R² Score: {privacy['r2_score']:.4f}")
    print(f"Correlation: {privacy['correlation']:.4f}")

## 6. Summary and Analysis

In [ ]:
# Create summary table
print("\n" + "="*80)
print("SUMMARY: Utility vs Privacy Trade-off")
print("="*80)

summary_data = []
for name, result in results.items():
    privacy = privacy_results[name]
    summary_data.append({
        'Method': name,
        'Accuracy': result['accuracy'],
        'F1': result['f1'],
        'AUC': result['auc'],
        'Privacy': privacy['privacy_score'],
        'R²': privacy['r2_score'],
        'Correlation': privacy['correlation']
    })

df_summary = pd.DataFrame(summary_data)
try:
    display(df_summary.round(4))
except NameError:
    print(df_summary.round(4).to_string())

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy vs Privacy
ax1 = axes[0]
for name, result in results.items():
    privacy = privacy_results[name]
    ax1.scatter(privacy['privacy_score'], result['accuracy'], label=name, s=100, alpha=0.7)

ax1.set_xlabel('Privacy Score (higher = better privacy)')
ax1.set_ylabel('Accuracy')
ax1.set_title('Privacy vs Utility Trade-off')
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.grid(True, alpha=0.3)

# R² vs Accuracy
ax2 = axes[1]
for name, result in results.items():
    privacy = privacy_results[name]
    ax2.scatter(privacy['r2_score'], result['accuracy'], label=name, s=100, alpha=0.7)

ax2.set_xlabel('R² Score (reconstruction quality, lower = better privacy)')
ax2.set_ylabel('Accuracy')
ax2.set_title('Reconstruction Quality vs Utility')
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("KEY INSIGHTS:")
print("="*80)
print("""
1. Random Walk Encoder applies linear transformations within the feature vector
2. A linear first layer in NN should be able to learn the inverse transformation
3. Privacy comes from the secret walk order and transformation weights
4. Binary features need special handling (preserve, normalize, or skip)
""")

## 7. Mathematical Analysis: Attack Complexity

### Brute-Force Attack Complexity

To reconstruct the original data, an attacker would need to:

1. **Guess the walk path**: The order of feature visits
2. **Guess the transformation weights**: The (a, b) pairs for each step

**Complexity Analysis:**

- **Walk path**: For n features and k steps, there are approximately n^k possible paths
- **Transformation weights**: For each step, (a, b) are sampled from a continuous distribution
  - If discretized to d levels each, there are d² possibilities per step
  - Total: (d²)^k possibilities

**Total brute-force complexity**: O(n^k × (d²)^k) = O((n × d²)^k)

For typical values:
- n = 7 features
- k = 14 steps (2 × n_features)
- d = 10 discretization levels

Complexity ≈ (7 × 100)^14 ≈ 10^39 operations

This is computationally infeasible for brute-force attacks.

### Linear Algebra Attack

However, if the attacker knows the transformation is linear, they could:

1. Model the transformation as: **X_encoded = T × X_original**
2. Try to learn the transformation matrix T from data
3. Invert: **X_original = T^(-1) × X_encoded**

**This is why we need:**
- Secret walk order (makes T non-obvious)
- Multiple steps (creates complex dependencies)
- Random weights (prevents simple matrix learning)

**Defense**: The combination of secret order + random weights makes it hard to learn T directly.

## 8. Analysis and Improvement Recommendations

### Key Findings from Results

1. **Encoded data outperforms baseline** (77.65% vs 64.25%): The linear mixing may act as implicit regularization, or the baseline MLP may be underfitting. **Stronger baseline** (100,50, max_iter=1000) and **multi-seed evaluation** (Section 4b) help check: if stronger baseline reaches ~77%, then encoding preserves rather than boosts; if not, encoding may add useful structure.

2. **Binary feature detection** (encoder fix): Only features with **exactly 2 unique values** (Sex, SibSp_bin, Parch_bin) are treated as binary. Pclass and Embarked (3 values) get the full walk.

3. **`skip binary` and `normalize binary` → zero privacy** (R²=1.0): **skip** leaves binary features in plaintext. **normalize** is fully linear → easy to invert. **Avoid both** when privacy matters.

4. **`preserve` is least bad for privacy** (R²≈0.96): Clipping to [0,1] adds non-linearity. **`output_noise`** (e.g. 0.08): Gaussian noise on outputs makes linear inversion harder; expect slightly lower R² and possibly better privacy with modest utility cost. Check privacy table for "Random Walk (output noise)".

5. **Transformation is linear and invertible**: Composition of `Xk = a*Xk + b*Xn` is linear (`X_enc = T @ X_orig`). With enough data, T can be learned. **output_noise** and **more steps** (denser T) are the main levers to harden.

### Recommendations

- **Utility**: `binary_handling='normalize'` or `'preserve'`, `weight_distribution='beta'`, `walk_strategy='random'`.
- **Privacy**: `binary_handling='preserve'`, `output_noise=0.05--0.1`, `n_steps=3*n_features` or more; **avoid** `skip` and `normalize`.
- **Stability**: Use **Section 4b (multi-seed)** to report mean±std. Prefer **stronger baseline** as reference when testing underfitting.
- **`display()`**: try/except fallback for non‑Jupyter.

## 9. Privacy Interpretation: R² and Statistical Significance

### Understanding R² in Privacy Context

**R² = 0.9377 means:**
- The attacker can explain **93.77%** of the variance in original features from encoded features
- **6.23%** of variance remains unexplained (1 - R² = 0.0623)

### Is R² = 0.9377 "Good" Privacy?

**For preventing exact reconstruction: YES, it's acceptable!**

- **Exact reconstruction** would require R² ≈ 1.0 (or very close)
- R² = 0.9377 means there's **6.23% unexplained variance** - the attacker cannot perfectly reconstruct
- The remaining variance represents **uncertainty/error** in reconstruction

### Statistical Significance Interpretation

**R² can be interpreted as a measure of information leakage:**

- **R² = 1.0**: Perfect reconstruction (0% privacy)
- **R² = 0.9377**: 93.77% information leaked, **6.23% privacy preserved**
- **R² = 0.5**: 50% information leaked, 50% privacy preserved
- **R² = 0.0**: No information leaked (100% privacy, but likely unusable for ML)

**The (1 - R²) value represents the "privacy level":**
- **Privacy Level = 1 - R² = 0.0623** (6.23% privacy)
- This means: **6.23% of the original information cannot be recovered** by the attacker

### Practical Interpretation

- **R² = 0.9377**: Attacker can reconstruct with **~6% error** (acceptable for preventing exact reconstruction)
- **R² = 0.99+**: Attacker can reconstruct with <1% error (very weak privacy)
- **R² = 0.90-0.95**: Attacker can reconstruct with 5-10% error (moderate privacy, acceptable for many use cases)
- **R² < 0.80**: Attacker has >20% error (strong privacy, but may hurt ML utility)

**Conclusion**: R² = 0.9377 is **acceptable** if the goal is preventing exact reconstruction, not perfect privacy.

In [ ]:
# Calculate privacy level from R²
print("="*80)
print("PRIVACY INTERPRETATION FROM R² SCORES")
print("="*80)

for name, result in results.items():
    privacy = privacy_results[name]
    r2 = privacy['r2_score']
    privacy_level = 1 - r2
    reconstruction_error_pct = privacy_level * 100
    
    print(f"\n{name}:")
    print(f"  R² = {r2:.4f}")
    print(f"  Privacy Level = 1 - R² = {privacy_level:.4f} ({reconstruction_error_pct:.2f}%)")
    print(f"  Interpretation: Attacker has {reconstruction_error_pct:.2f}% error in reconstruction")
    
    if r2 < 0.80:
        privacy_rating = "STRONG"
    elif r2 < 0.90:
        privacy_rating = "MODERATE"
    elif r2 < 0.95:
        privacy_rating = "ACCEPTABLE (prevents exact reconstruction)"
    elif r2 < 0.99:
        privacy_rating = "WEAK"
    else:
        privacy_rating = "VERY WEAK"
    
    print(f"  Privacy Rating: {privacy_rating}")

print("\n" + "="*80)
print("KEY INSIGHT:")
print("="*80)
print("""
For preventing EXACT reconstruction (your goal):
- R² < 0.95 is ACCEPTABLE (attacker has >5% error)
- R² = 0.9377 means 6.23% privacy → attacker cannot exactly reconstruct
- The (1-R²) value represents the 'privacy level' or 'reconstruction uncertainty'
""")

## 10. Comparison with Other Encoding Methods

Compare Random Walk encoder with other methods from the codebase to see which provides the best accuracy/privacy trade-off.

In [ ]:
# Import other encoders for comparison
from quagua.encoders import (
    ChaoticSystemEncoder,
    QuantumWalkEncoder,
    QuantumEntanglementEncoder,
    ClassicalXORMixer,
    OrthogonalSubspaceEncoder,
    FastPrivacyEncoder,
    EncodeThenMixPipeline,
)

# Define comparison methods (best performers from sds.ipynb)
comparison_methods = {
    'Baseline (No Encoding)': lambda x: x,
    'Chaotic (Logistic) + Quantum Walk': EncodeThenMixPipeline(
        ChaoticSystemEncoder(map_type='logistic', n_iterations=5),
        QuantumWalkEncoder(n_steps=3, graph_type='complete')
    ),
    'Chaotic (Logistic) + Entanglement': EncodeThenMixPipeline(
        ChaoticSystemEncoder(map_type='logistic', n_iterations=5),
        QuantumEntanglementEncoder(entanglement_strength=0.4)
    ),
    'Orthogonal + Quantum Walk': EncodeThenMixPipeline(
        OrthogonalSubspaceEncoder(subspace_dim=3),
        QuantumWalkEncoder(n_steps=3, graph_type='complete')
    ),
    'Quantum Walk (standalone)': QuantumWalkEncoder(n_steps=3, graph_type='complete'),
    'Fast Privacy (default)': FastPrivacyEncoder(
        encoding_dim=4, polynomial_degree=1, noise_level=0.1, use_mixing=True, hash_seed=42
    ),
    # Add best Random Walk variants
    'Random Walk (default)': RandomWalkFeatureEncoder(
        n_steps=None, weight_distribution='beta', walk_strategy='random',
        binary_handling='preserve', seed=42
    ),
    'Random Walk (output noise)': RandomWalkFeatureEncoder(
        n_steps=None, weight_distribution='beta', walk_strategy='random',
        binary_handling='preserve', output_noise=0.08, seed=42
    ),
    'Random Walk (more steps)': RandomWalkFeatureEncoder(
        n_steps=3*X_train.shape[1], weight_distribution='beta', walk_strategy='random',
        binary_handling='preserve', seed=42
    ),
}

print(f"Comparing {len(comparison_methods)} methods...")
print("="*80)

In [ ]:
# Evaluate all comparison methods
comparison_results = {}
comparison_privacy = {}

framework = PrivacyAITestFramework()

for name, encoder in comparison_methods.items():
    print(f"\n{'='*80}")
    print(f"Evaluating: {name}")
    print(f"{'='*80}")
    
    # Encode data
    if callable(encoder) and not hasattr(encoder, 'fit'):
        # Lambda (baseline) - no transformation
        X_train_enc = encoder(X_train)
        X_test_enc = encoder(X_test)
        
        class BaselineWrapper:
            def transform(self, X):
                return X  # Identity transformation
        enc_wrapper = BaselineWrapper()
    else:
        # Encoder with fit/transform
        X_train_enc = encoder.fit_transform(X_train)
        X_test_enc = encoder.transform(X_test)
        
        class EncWrapper:
            def __init__(self, enc):
                self.encoder = enc
            def transform(self, X):
                return self.encoder.transform(X)
        enc_wrapper = EncWrapper(encoder)
    
    # Train enhanced models (same as baseline and Random Walk tests)
    model_results = train_models(X_train_enc, y_train, X_test_enc, y_test, verbose=False)
    
    # Use best model results
    best_name = model_results['best_model_name']
    best_results = model_results['all_results'][best_name]
    
    acc = best_results['accuracy']
    f1 = best_results['f1']
    auc = best_results['auc']
    
    # Evaluate privacy
    privacy = framework.evaluate_privacy(
        encoder=enc_wrapper,
        X_test=X_test,
        encoder_name=name
    )
    
    comparison_results[name] = {
        'accuracy': acc,
        'f1': f1,
        'auc': auc,
        'best_model_name': best_name,
        'all_models': model_results['all_results']
    }
    comparison_privacy[name] = privacy
    
    print(f"  Best Model: {best_name}")
    print(f"  Accuracy: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")
    print(f"  Privacy: {privacy['privacy_score']:.4f}, R²: {privacy['r2_score']:.4f}, Privacy Level: {1-privacy['r2_score']:.4f}")

In [ ]:
# Create comparison table
print("\n" + "="*80)
print("COMPREHENSIVE COMPARISON: All Methods")
print("="*80)

comp_data = []
for name in comparison_results:
    comp_data.append({
        'Method': name,
        'Accuracy': comparison_results[name]['accuracy'],
        'F1': comparison_results[name]['f1'],
        'AUC': comparison_results[name]['auc'],
        'Privacy Score': comparison_privacy[name]['privacy_score'],
        'R²': comparison_privacy[name]['r2_score'],
        'Privacy Level (1-R²)': 1 - comparison_privacy[name]['r2_score'],
        'Correlation': comparison_privacy[name]['correlation']
    })

df_comp = pd.DataFrame(comp_data)
df_comp = df_comp.sort_values('Accuracy', ascending=False)

try:
    display(df_comp.round(4))
except NameError:
    print(df_comp.round(4).to_string())

# Find Pareto-optimal methods (best accuracy for given privacy level)
print("\n" + "="*80)
print("PARETO-OPTIMAL METHODS (Best Accuracy for Privacy Level)")
print("="*80)

# Group by privacy level ranges and find best accuracy in each
privacy_ranges = [
    (0.0, 0.05, "Very Weak (R² > 0.95)"),
    (0.05, 0.10, "Weak (R² 0.90-0.95)"),
    (0.10, 0.20, "Moderate (R² 0.80-0.90)"),
    (0.20, 1.0, "Strong (R² < 0.80)"),
]

for min_priv, max_priv, label in privacy_ranges:
    mask = (df_comp['Privacy Level (1-R²)'] >= min_priv) & (df_comp['Privacy Level (1-R²)'] < max_priv)
    if mask.any():
        best = df_comp[mask].nlargest(1, 'Accuracy')
        if len(best) > 0:
            row = best.iloc[0]
            print(f"\n{label}:")
            print(f"  Best: {row['Method']}")
            print(f"    Accuracy: {row['Accuracy']:.4f}, Privacy Level: {row['Privacy Level (1-R²)']:.4f}, R²: {row['R²']:.4f}")

# Find methods that beat Random Walk
print("\n" + "="*80)
print("METHODS BETTER THAN RANDOM WALK (default)")
print("="*80)

rw_default_acc = comparison_results['Random Walk (default)']['accuracy']
rw_default_priv = 1 - comparison_privacy['Random Walk (default)']['r2_score']

print(f"Random Walk (default) baseline:")
print(f"  Accuracy: {rw_default_acc:.4f}, Privacy Level: {rw_default_priv:.4f}")

better_methods = []
for name in comparison_results:
    if name == 'Random Walk (default)':
        continue
    acc = comparison_results[name]['accuracy']
    priv = 1 - comparison_privacy[name]['r2_score']
    
    # Better if: (higher accuracy AND similar/better privacy) OR (similar accuracy AND better privacy)
    if (acc >= rw_default_acc * 0.98 and priv >= rw_default_priv * 0.9) or \
       (acc >= rw_default_acc * 0.95 and priv > rw_default_priv * 1.1):
        better_methods.append((name, acc, priv))

if better_methods:
    print(f"\nMethods that are competitive or better:")
    for name, acc, priv in sorted(better_methods, key=lambda x: x[1], reverse=True):
        print(f"  {name}: Acc={acc:.4f}, Privacy Level={priv:.4f}")
else:
    print("\nNo methods clearly better than Random Walk (default)")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy vs Privacy Level
ax1 = axes[0]
for name in comparison_results:
    acc = comparison_results[name]['accuracy']
    priv = 1 - comparison_privacy[name]['r2_score']
    marker = 'o' if 'Random Walk' in name else 's'
    size = 150 if 'Random Walk' in name else 100
    alpha = 0.8 if 'Random Walk' in name else 0.6
    ax1.scatter(priv, acc, label=name, s=size, marker=marker, alpha=alpha)

ax1.set_xlabel('Privacy Level (1 - R², higher = better privacy)')
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy vs Privacy Level Trade-off\n(All Methods)')
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax1.grid(True, alpha=0.3)

# R² vs Accuracy
ax2 = axes[1]
for name in comparison_results:
    acc = comparison_results[name]['accuracy']
    r2 = comparison_privacy[name]['r2_score']
    marker = 'o' if 'Random Walk' in name else 's'
    size = 150 if 'Random Walk' in name else 100
    alpha = 0.8 if 'Random Walk' in name else 0.6
    ax2.scatter(r2, acc, label=name, s=size, marker=marker, alpha=alpha)

ax2.set_xlabel('R² Score (reconstruction quality, lower = better privacy)')
ax2.set_ylabel('Accuracy')
ax2.set_title('Reconstruction Quality (R²) vs Accuracy\n(All Methods)')
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Final Analysis: Privacy Interpretation and Method Comparison

### Key Findings

1. **Privacy Interpretation (R² = 0.9377)**:
   - **Privacy Level = 1 - R² = 0.0623** (6.23% privacy)
   - This means the attacker has **6.23% reconstruction error** - they cannot exactly reconstruct
   - **For preventing exact reconstruction: This is ACCEPTABLE!**
   - The goal is not perfect privacy (R² = 0), but preventing exact reconstruction (R² < 0.95)

2. **Statistical Significance**:
   - **R² = 0.9377** means the attacker can explain 93.77% of variance
   - The remaining **6.23% is unexplained variance** = reconstruction uncertainty
   - This uncertainty prevents exact reconstruction, which is the goal

3. **Method Comparison**:
   - Check the comparison table above to see which methods have:
     - Higher accuracy than Random Walk
     - Similar or better privacy level (1-R²)
   - Look for Pareto-optimal methods (best accuracy for each privacy level range)

### Recommendations

- **If goal is preventing exact reconstruction**: R² < 0.95 (Privacy Level > 0.05) is acceptable
- **If goal is strong privacy**: Aim for R² < 0.80 (Privacy Level > 0.20), but expect lower accuracy
- **Best trade-off**: Look for methods with Accuracy > 0.75 and Privacy Level > 0.05